# Certify a raw-score false-alarm rate

`FPRCertificate` answers a practical question: how high should the anomaly-score threshold be so that at most 5% of future clean inliers are flagged? It uses clean calibration scores and does not need a test batch to build the certificate.

This is different from `fdp_bounds()`: FPR controls false alarms for future inliers, while FDP certification describes the false-discovery proportion of one fixed batch.

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

from nonconform import ConformalDetector, Split

## Fit a detector on clean reference data

The shifted points in `x_test` are included only to make the resulting mask interesting. They are not used to construct the certificate.

In [ ]:
rng = np.random.default_rng(42)
x_reference = rng.normal(size=(3_000, 2))
x_test = np.vstack([rng.normal(size=(95, 2)), rng.normal(loc=5.0, size=(5, 2))])

detector = ConformalDetector(
    detector=IsolationForest(n_estimators=100, random_state=42),
    strategy=Split(n_calib=1_000),
    seed=42,
).fit(x_reference)

## Choose a threshold with a certified target FPR

`confidence=0.95` is the coverage of the whole threshold curve. `target_fpr=0.05` is the operational false-alarm target. The Monte Carlo randomness is used once when the certificate is created; later queries do not resample.

In [ ]:
certificate = detector.fpr_bounds(
    confidence=0.95,
    n_resamples=1_000,
    seed=42,
)

target_fpr = 0.05
threshold = certificate.threshold_for(target_fpr)
scores = detector.score_samples(x_test)
mask = certificate.select(scores, target_fpr=target_fpr)

print(f"Certified threshold: {threshold}")
print(f"Selected observations: {int(mask.sum())}")
print(certificate.to_frame(np.array([threshold])).to_string(index=False))

The selected mask uses the inclusive rule `score >= threshold`, with larger normalized scores meaning more anomalous. The guarantee is about the marginal false-alarm rate of a future clean inlier; it does not guarantee that 5% of every finite test batch are false alarms, and it says nothing about recall.